# Carnet de démarrage — sujet 02 · Favorita

**Objectif du sujet** : prévoir les ventes d'un article, par magasin, sur 16 jours.

Ce carnet est un **châssis**, pas une solution. Il fait quatre choses et s'arrête :

1. il **trouve** vos fichiers, quelle que soit la plateforme ;
2. il **charge** la table efficacement — cache Parquet construit une fois, projection de colonnes ;
3. il **profile** les données : dimensions, types, cible, manquants ;
4. il **écrit** une soumission au bon format, avec une valeur de référence volontairement nulle.

**Ce qu'il ne fait pas** : choisir votre métrique, votre découpage de validation, votre modèle, ni vos
variables. Ces décisions sont le sujet, et c'est ce que la grille note.

12,2 M de lignes. `horizon.csv` est une grille dense, sans cible. Les ventes négatives sont des retours.

---

## Où mettre ce carnet

| Plateforme | Ce qu'il faut faire |
|---|---|
| **Le dossier du sujet** | **https://drive.google.com/drive/folders/1N1EJk5sY8cJ3PFtipJFbMFJIG2AqUYYl?usp=sharing** — téléchargez-le avant tout (le lien figure aussi dans l'énoncé) |
| **En local (VS Code, Jupyter)** | placez le carnet **dans le dossier téléchargé**, ou définissez `DONNEES_PROJET` vers ce dossier |
| **Google Colab** | déposez les fichiers dans `MyDrive/Sujet_02_Favorita/` — le carnet les trouvera ; ou téléversez-les dans `/content/` |
| **Kaggle** | créez un jeu de données avec vos fichiers, **Add Input** — `/kaggle/input` est exploré automatiquement |

**Rien n'est codé en dur** : le carnet cherche, et s'il ne trouve pas, il **dit précisément** ce qu'il a
cherché et où.


## Les fichiers de ce sujet

| Fichier | Rôle |
|---|---|
| `train.csv` | entraînement ou annexe |
| `horizon.csv` | à prédire — la cible est absente |
| `items.csv` | entraînement ou annexe |
| `stores.csv` | entraînement ou annexe |
| `holidays_events.csv` | entraînement ou annexe |
| `oil.csv` | entraînement ou annexe |
| `transactions.csv` | entraînement ou annexe |

**Format de sortie attendu** : `id,unit_sales`.
**Colonne d'identifiant** : `id`.
**Fichier à produire** : `soumission.csv`, à la racine de votre dépôt.


## Sur les données lourdes — et pourquoi pas « raylib »

`raylib` est une bibliothèque **graphique** (fenêtres, dessin, jeux) : elle ne lit aucune table. Elle
n'a rien à faire ici. Les outils qui répondent au besoin sont :

| Outil | Rôle | Présent par défaut ? |
|---|---|---|
| **Parquet + pyarrow** | format colonne compressé, lecture sélective | **oui** avec pandas récent |
| **Polars** | moteur colonne, `scan_csv` paresseux | non — `pip install polars` |
| **DuckDB** | SQL directement sur CSV/Parquet, sans chargement | non — `pip install duckdb` |

Ce carnet utilise **Parquet** (garanti), et le remplace par Polars ou DuckDB si vous les installez. Le
principe ne change pas : **ne jamais charger ce dont on n'a pas besoin**, et **payer la conversion
CSV → Parquet une seule fois**.


In [1]:
# --- 1. Réglages --------------------------------------------------------------
from pathlib import Path
import os
import pandas as pd

# Où télécharger vos données — le dossier de votre sujet sur le Drive du cours.
LIEN_DRIVE = "__LIEN__"

# ECHANTILLON : nombre de lignes lues pour démarrer. None = tout lire.
ECHANTILLON = 200_000
DOSSIER_CACHE = Path.cwd() / "_cache_parquet"


In [2]:
from pathlib import Path
print("dossier courant :", Path.cwd())
print("contenu :", sorted(p.name for p in Path.cwd().iterdir())[:30])

dossier courant : /home/gille/favorita-projet/membres/Gille
contenu : ['Favorita_P2_etapes.ipynb', 'Gille_favorita.ipynb', '_cache_parquet', 'soumission.csv']


In [3]:
import os
from pathlib import Path

dossier = Path.cwd() / "Projet_app_auto"
print(dossier.exists(), sorted(p.name for p in dossier.iterdir())[:12] if dossier.exists() else "")
os.environ["DONNEES_PROJET"] = str(dossier)

False 


In [4]:
import os
os.environ["DONNEES_PROJET"] = "/home/gille/Documents/cours GI_L1/GI_L2/esgc-vak_apprentissage-automatique/carnets/Projet_app_auto"

In [5]:
# --- 2. Trouver les données, sans rien coder en dur --------------------------
FICHIERS_ATTENDUS = ['train.csv', 'horizon.csv', 'items.csv', 'stores.csv', 'holidays_events.csv', 'oil.csv', 'transactions.csv']
SUJET = "Sujet_02_Favorita"


def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    """Cherche le dossier qui contient TOUS les fichiers attendus."""
    essais = []
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]

    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Dossier de données introuvable.\n"
        "Il faut un dossier contenant : " + ", ".join(fichiers) + "\n"
        "Solutions : placez le carnet à côté de ce dossier, "
        "ou définissez DONNEES_PROJET=/chemin/vers/le/dossier.\n"
        "Chemins essayés : " + str(essais[:12])
    )


DOSSIER = trouver_donnees()
print("dossier de données :", DOSSIER)
print("fichiers vus :", sorted(p.name for p in DOSSIER.iterdir() if p.is_file())[:12])


dossier de données : /home/gille/Documents/cours GI_L1/GI_L2/esgc-vak_apprentissage-automatique/carnets/Projet_app_auto
fichiers vus : ['Carnet_de_demarrage.ipynb', 'Enonce.md', 'holidays_events.csv', 'horizon.csv', 'items.csv', 'oil.csv', 'soumission.csv', 'stores.csv', 'train.csv', 'transactions.csv']


In [6]:
# --- 3. Chargement efficace (table lourde) ------------------------------------
# Le cache Parquet s'écrit à côté du carnet, jamais dans les données : sur Kaggle,
# /kaggle/input est en lecture seule.
import pyarrow as pa
import pyarrow.csv as pvc
import pyarrow.parquet as pq


def table(nom, colonnes=None, complet=False):
    """Ouvre un CSV en profitant d'un cache Parquet.

    nom      : nom du fichier dans le dossier de données
    colonnes : liste de colonnes à garder (None = toutes) — le levier principal
    morceaux : si un entier, rend un itérateur de DataFrames de cette taille
    """
    source = DOSSIER / nom
    if not source.is_file():
        raise FileNotFoundError(f"{nom} introuvable dans {DOSSIER}")

    # lecture d'un échantillon : pas de cache, on veut seulement voir
    if ECHANTILLON and not complet:
        lots, reste = [], ECHANTILLON
        for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 25)):
            lots.append(lot)
            reste -= lot.num_rows
            if reste <= 0:
                break
        t = pa.Table.from_batches(lots)
        if colonnes:
            t = t.select([c for c in colonnes if c in t.column_names])
        return t.to_pandas()

    # cache Parquet, construit une fois
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    if not cache.is_file():
        DOSSIER_CACHE.mkdir(parents=True, exist_ok=True)
        print(f"construction du cache Parquet : {cache.name} (une seule fois)")
        ecrivain = None
        try:
            for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 26)):
                if colonnes:
                    lot = lot.select([c for c in colonnes if c in lot.schema.names])
                if ecrivain is None:
                    ecrivain = pq.ParquetWriter(cache, lot.schema)
                ecrivain.write_batch(lot)
        finally:
            if ecrivain is not None:
                ecrivain.close()
    fichier = pq.ParquetFile(cache)
    return fichier.read(columns=colonnes).to_pandas()


def table_par_morceaux(nom, colonnes=None, morceaux=200_000):
    """Itère sur la table par paquets — pour les traitements qui ne tiennent pas en mémoire."""
    table(nom, colonnes=colonnes)          # garantit que le cache existe
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    for lot in pq.ParquetFile(cache).iter_batches(batch_size=morceaux):
        yield lot.to_pandas()


In [7]:
# --- 4. Charger et regarder ---------------------------------------------------
entrainement = table("train.csv")
print("dimensions :", entrainement.shape)
print("colonnes :", len(entrainement.columns))
entrainement.head()


dimensions : (954613, 6)
colonnes : 6


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,9,2013-01-01,25,114790,3.0,None
1,10,2013-01-01,25,114800,1.0,None
2,14,2013-01-01,25,115720,5.0,None
3,15,2013-01-01,25,115850,1.0,None
4,30,2013-01-01,25,158680,1.0,None


In [8]:
# --- 5. Profil de la cible ----------------------------------------------------
CIBLE = "unit_sales"
if CIBLE in entrainement.columns:
    serie = entrainement[CIBLE]
    print("type :", serie.dtype)
    print("manquants :", int(serie.isna().sum()))
    print(serie.describe())
    if serie.nunique() <= 20:
        print(serie.value_counts(normalize=True).sort_index())
else:
    print(f"la cible {CIBLE} est absente de cette table — normal pour la table à prédire")


type : float64
manquants : 0
count    954613.000000
mean          8.840643
std          22.341767
min        -149.000000
25%           2.000000
50%           4.000000
75%           9.000000
max       10063.000000
Name: unit_sales, dtype: float64


In [9]:
# --- 6. Soumission de RÉFÉRENCE (format seulement) ----------------------------
# Cette valeur n'a AUCUNE prétention prédictive : elle vérifie le format.
COLONNES_SORTIE = ['id', 'unit_sales']


def ecrire_soumission(identifiants, valeurs=None, chemin="soumission.csv"):
    """Écrit le fichier attendu. `valeurs` peut être une colonne, ou une matrice de probabilités."""
    sortie = pd.DataFrame({"id": identifiants})
    if len(COLONNES_SORTIE) == 2:
        sortie["unit_sales"] = valeurs
    else:
        n = len(COLONNES_SORTIE) - 1
        matrice = valeurs if valeurs is not None else [[1.0 / n] * n for _ in identifiants]
        for j, colonne in enumerate(COLONNES_SORTIE[1:]):
            sortie[colonne] = [ligne[j] for ligne in matrice]
    sortie.to_csv(chemin, index=False)
    print(f"écrit : {chemin} · {len(sortie):,} lignes · colonnes {list(sortie.columns)}")
    if len(COLONNES_SORTIE) > 2:
        print("somme par ligne (doit valoir 1) :", sortie.iloc[:, 1:].sum(axis=1).round(6).unique()[:3])
    return sortie


a_predire = table("horizon.csv", colonnes=["id"], complet=True)
identifiants = a_predire["id"].drop_duplicates().to_numpy()
print("identifiants à prédire :", len(identifiants))
ecrire_soumission(identifiants)


identifiants à prédire : 277568
écrit : soumission.csv · 277,568 lignes · colonnes ['id', 'unit_sales']


,id,unit_sales
0,1,None
1,2,None
2,3,None
3,4,None
4,5,None
...,...,...
277563,277564,None
277564,277565,None
277565,277566,None
277566,277567,None


In [10]:
# --- 7. Le travail commence ICI -----------------------------------------------
# À vous de décider, et de justifier :
#   - la métrique, et ce qu'elle coûte ;
#   - le découpage de validation ;
#   - les variables, leur encodage, leur imputation ;
#   - le modèle, son réglage, son seuil.
# Ce carnet ne le fera pas à votre place : c'est l'objet du projet.
#raise NotImplementedError(
#    "Carnet de démarrage : la modélisation est à votre charge. Écrivez votre pipeline à partir d'ici."
#)


# Favorita — Sujet 02 · Parcours P2

**Étapes assignées :** 1.2 Charger et compter → 1.3 Typer → 1.5 Valeurs impossibles → 1.10 Le plancher

**Ordre de livraison :** pousse **1.2 et 1.3 en premier** : P3 (1.4) attend ta 1.3, et P1 (1.8) a besoin de tables chargées et typées.

Les noms de variables sont ceux de l'équipe, pour que les cellules de P1, P3 et P4 fonctionnent après les tiennes :
`train, horizon, items, stores, holidays, oil, transactions, tables`.

Chaque étape suit le même schéma : **ce qu'on fait et pourquoi → le code commenté → ce que tu dois noter dans le rendu**.

## 1.2 · Charger et compter

**Ce qu'on fait :** lire les 7 fichiers CSV et dire combien de lignes et de colonnes contient chaque table.

**Pourquoi :** toutes les étapes suivantes partent de ces tables ; il faut aussi vérifier que rien n'a été tronqué à la lecture.

**Fonctions utilisées :**
- `pd.read_csv(chemin)` lit un CSV et rend un tableau (`DataFrame`).
- `parse_dates=["date"]` demande à pandas de lire cette colonne comme de vraies dates, pas comme du texte.
- `tableau.shape` rend `(nombre de lignes, nombre de colonnes)`.
- Un **dictionnaire** `tables = {nom: tableau}` permet de parcourir toutes les tables avec une seule boucle `for`.


In [11]:
# --- P2 · 1.2 Charger et compter ---------------------------------------------
train        = pd.read_csv(DOSSIER / "train.csv",   parse_dates=["date"])
horizon      = pd.read_csv(DOSSIER / "horizon.csv", parse_dates=["date"])
items        = pd.read_csv(DOSSIER / "items.csv")
stores       = pd.read_csv(DOSSIER / "stores.csv")
holidays     = pd.read_csv(DOSSIER / "holidays_events.csv", parse_dates=["date"])
oil          = pd.read_csv(DOSSIER / "oil.csv",          parse_dates=["date"])
transactions = pd.read_csv(DOSSIER / "transactions.csv", parse_dates=["date"])

# Un dictionnaire pour parcourir toutes les tables avec une boucle
tables = {"train": train, "horizon": horizon, "items": items, "stores": stores,
          "holidays": holidays, "oil": oil, "transactions": transactions}

print(f"{'table':13s} {'lignes':>12s}   colonnes")
for nom, tableau in tables.items():
    nb_lignes, nb_colonnes = tableau.shape
    print(f"{nom:13s} {nb_lignes:>12,}   {nb_colonnes}   {list(tableau.columns)}")

table               lignes   colonnes
train           12,248,320   6   ['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion']
horizon            277,568   5   ['id', 'date', 'store_nbr', 'item_nbr', 'onpromotion']
items                4,100   4   ['item_nbr', 'family', 'class', 'perishable']
stores                  54   5   ['store_nbr', 'city', 'state', 'type', 'cluster']
holidays               350   6   ['date', 'type', 'locale', 'locale_name', 'description', 'transferred']
oil                  1,218   2   ['date', 'dcoilwtico']
transactions        83,488   3   ['date', 'store_nbr', 'transactions']


## 1.3 · Typer

**Ce qu'on fait :** donner à chaque colonne le bon *type* (entier, décimal, texte, date, oui/non), et mesurer la mémoire gagnée.

**Pourquoi :** par défaut pandas prend de gros types (`int64`, `float64`, `object`). Un petit type prend moins de mémoire et évite des erreurs. Sur 12 M de lignes, la différence se compte en centaines de Mo.

**Fonctions utilisées :**
- `tableau.dtypes` affiche le type de chaque colonne.
- `colonne.astype("int16")` convertit une colonne dans un autre type.
- `pd.to_datetime(colonne)` convertit du texte en dates.
- `"category"` : type pour le texte qui se répète beaucoup (famille, ville…) ; chaque valeur n'est stockée qu'une fois.
- `"boolean"` : oui/non qui accepte les cases vides (`True`, `False`, `<NA>`).
- `tableau.memory_usage(deep=True).sum()` donne la mémoire en octets.

**Choix de types :** `store_nbr` (1 à 54) tient dans `int16` ; `item_nbr` (identifiants jusqu'à quelques millions) dans `int32` ; `onpromotion` est un oui/non, pas un nombre à virgule.

In [13]:
# --- P2 · 1.3 Typer -----------------------------------------------------------
# ÉTAPE 1 : photographier la situation AVANT correction
memoire_avant = {nom: t.memory_usage(deep=True).sum() / 1e6 for nom, t in tables.items()}
print("=== Types AVANT correction ===")
for nom, tableau in tables.items():
    print("---", nom)
    print(tableau.dtypes.to_string())

print("\nonpromotion, valeurs présentes dans train :")
print(train["onpromotion"].value_counts(dropna=False).head())

# ÉTAPE 2 : corriger
# 2a) Les dates : si une colonne "date" est encore du texte, on la convertit.
for tableau in (train, horizon, holidays, oil, transactions):
    tableau["date"] = pd.to_datetime(tableau["date"])

# 2b) onpromotion : un oui/non. 0.0 -> False, 1.0 -> True, case vide -> <NA>.
train["onpromotion"]   = train["onpromotion"].astype("boolean")
horizon["onpromotion"] = horizon["onpromotion"].astype("boolean")

# 2c) Les identifiants : de petits entiers prennent moins de place.
for tableau in (train, horizon):
    tableau["store_nbr"] = tableau["store_nbr"].astype("int16")
    tableau["item_nbr"]  = tableau["item_nbr"].astype("int32")
items["item_nbr"]   = items["item_nbr"].astype("int32")
stores["store_nbr"] = stores["store_nbr"].astype("int16")
transactions["store_nbr"] = transactions["store_nbr"].astype("int16")

# 2d) Le texte qui se répète beaucoup : type "category".
items["family"] = items["family"].astype("category")
for colonne in ("city", "state", "type"):
    stores[colonne] = stores[colonne].astype("category")

# 2e) La cible : float32 suffit (précision d'environ 7 chiffres).
train["unit_sales"] = train["unit_sales"].astype("float32")

# ÉTAPE 3 : vérifier APRÈS correction
print("\n=== Types APRÈS correction (train) ===")
print(train.dtypes.to_string())
print("\n=== Mémoire (Mo) ===")
for nom, tableau in tables.items():
    apres = tableau.memory_usage(deep=True).sum() / 1e6
    print(f"{nom:13s} {memoire_avant[nom]:>9.1f} -> {apres:>9.1f}")

=== Types AVANT correction ===
--- train
id                      int64
date           datetime64[us]
store_nbr               int16
item_nbr                int32
unit_sales            float32
onpromotion           boolean
--- horizon
id                      int64
date           datetime64[us]
store_nbr               int16
item_nbr                int32
onpromotion           boolean
--- items
item_nbr         int32
family        category
class            int64
perishable       int64
--- stores
store_nbr       int16
city         category
state        category
type         category
cluster         int64
--- holidays
date           datetime64[us]
type                      str
locale                    str
locale_name               str
description               str
transferred              bool
--- oil
date          datetime64[us]
dcoilwtico           float64
--- transactions
date            datetime64[us]
store_nbr                int16
transactions             int64

onpromotion, valeurs pré


=== Types APRÈS correction (train) ===
id                      int64
date           datetime64[us]
store_nbr               int16
item_nbr                int32
unit_sales            float32
onpromotion           boolean

=== Mémoire (Mo) ===
train             343.0 ->     343.0
horizon             6.7 ->       6.7
items               0.1 ->       0.1
stores              0.0 ->       0.0
holidays            0.0 ->       0.0
oil                 0.0 ->       0.0
transactions        1.5 ->       1.5


## 1.5 · Valeurs impossibles

**Ce qu'on fait :** chercher des valeurs qui ne devraient pas exister, et distinguer ce qui est **anormal** de ce qui est **inhabituel mais normal**.

**Pourquoi :** une valeur impossible fausse les moyennes et les modèles. Mais en supprimer une qui est en fait normale (un retour, un article vendu au poids) fausserait aussi.

**Fonctions utilisées :**
- `(colonne < 0)` donne vrai/faux pour chaque ligne ; `.sum()` compte les « vrai » ; `.mean()` donne la proportion.
- `colonne.isna()` repère les cases vides.
- `colonne % 1 != 0` repère les valeurs non entières.
- `colonne.isin(autre_colonne)` teste si chaque valeur existe dans une table de référence ; le `~` signifie « le contraire de ». Un résultat de **0** est le bon résultat.
- `colonne.quantile(0.9999)` donne la valeur sous laquelle se trouvent 99,99 % des lignes (utile pour repérer les extrêmes).

**À savoir :** les ventes **négatives** sont des retours (le sujet le dit), donc pas des erreurs. Les ventes **non entières** sont normales pour les articles vendus au poids.

In [14]:
# --- P2 · 1.5 Valeurs impossibles ---------------------------------------------
total = len(train)

# ÉTAPE 1 : la cible unit_sales
print("=== 1. La cible unit_sales (train) ===")
y = train["unit_sales"]
print("manquants        :", int(y.isna().sum()))
print(f"négatifs (< 0)   : {(y < 0).sum():,}  ({(y < 0).sum() / total:.4%})   <- retours")
print(f"égaux à zéro     : {(y == 0).sum():,}  ({(y == 0).sum() / total:.4%})")
print(f"non entiers      : {((y % 1) != 0).sum():,}   <- normal pour les articles vendus au poids")
print("minimum          :", y.min())
print("maximum          :", y.max())
print("quantile 99,99 % :", y.quantile(0.9999))

# ÉTAPE 2 : la promotion
print("\n=== 2. onpromotion ===")
print("train   manquants :", int(train["onpromotion"].isna().sum()))
print("horizon manquants :", int(horizon["onpromotion"].isna().sum()))

# ÉTAPE 3 : les identifiants existent-ils dans les tables de référence ?
print("\n=== 3. Identifiants inconnus (0 = bon résultat) ===")
print("magasins inconnus (train)   :", int((~train["store_nbr"].isin(stores["store_nbr"])).sum()))
print("articles inconnus (train)   :", int((~train["item_nbr"].isin(items["item_nbr"])).sum()))
print("magasins inconnus (horizon) :", int((~horizon["store_nbr"].isin(stores["store_nbr"])).sum()))
print("articles inconnus (horizon) :", int((~horizon["item_nbr"].isin(items["item_nbr"])).sum()))

# ÉTAPE 4 : les annexes
print("\n=== 4. Annexes ===")
print("pétrole : prix <= 0   :", int((oil["dcoilwtico"] <= 0).sum()))
print("pétrole : prix vides  :", int(oil["dcoilwtico"].isna().sum()))
print("transactions <= 0     :", int((transactions["transactions"] <= 0).sum()))

# ÉTAPE 5 : les plages de dates
print("\n=== 5. Dates ===")
for nom, tableau in tables.items():
    if "date" in tableau.columns:
        print(f"{nom:13s} de {tableau['date'].min().date()} à {tableau['date'].max().date()}"
              f"   ({tableau['date'].nunique()} jours différents)")

=== 1. La cible unit_sales (train) ===
manquants        : 0
négatifs (< 0)   : 853  (0.0070%)   <- retours
égaux à zéro     : 0  (0.0000%)
non entiers      : 774,715   <- normal pour les articles vendus au poids
minimum          : -10002.0
maximum          : 20748.0
quantile 99,99 % : 568.1680999994278

=== 2. onpromotion ===
train   manquants : 12248320
horizon manquants : 0

=== 3. Identifiants inconnus (0 = bon résultat) ===
magasins inconnus (train)   : 0
articles inconnus (train)   : 0
magasins inconnus (horizon) : 0
articles inconnus (horizon) : 0

=== 4. Annexes ===
pétrole : prix <= 0   : 0
pétrole : prix vides  : 43
transactions <= 0     : 0

=== 5. Dates ===
train         de 2013-01-01 à 2017-07-30   (1668 jours différents)
horizon       de 2017-07-31 à 2017-08-15   (16 jours différents)
holidays      de 2012-03-02 à 2017-12-26   (312 jours différents)
oil           de 2013-01-01 à 2017-08-31   (1218 jours différents)
transactions  de 2013-01-01 à 2017-08-15   (1682 jours dif

## 1.10 · Le plancher

**Ce qu'on fait :** mesurer l'erreur de **règles très simples**. Ce « plancher » est le score à battre : un modèle qui ne fait pas mieux n'apporte rien.

**Pourquoi un découpage dans le temps :** on prédit le futur, donc on calcule les règles sur le **passé** et on les teste sur les **16 derniers jours** de `train`. Un tirage au hasard laisserait le modèle « voir » le futur.

**Les règles, de la plus bête à la plus fine :**
0. toujours prédire 0 (c'est la soumission de référence du carnet de départ) ;
1. la moyenne de toutes les ventes du passé ;
2. la moyenne de l'**article** ;
3. la moyenne du **couple (magasin, article)** ;
4. la moyenne du couple **le même jour de la semaine**, sur les 4 dernières semaines.

Quand une règle n'a rien à dire (couple jamais vu), elle se rabat sur la règle précédente (`.fillna(...)`).



In [16]:
import numpy as np

In [17]:
# --- P2 · 1.10 Le plancher ----------------------------------------------------
# ÉTAPE 1 : découper dans le temps (jamais au hasard)
fin_train   = train["date"].max()
debut_valid = fin_train - pd.Timedelta(days=15)           # 16 derniers jours

passe = train[train["date"] < debut_valid].copy()         # sert à calculer les règles
valid = train[train["date"] >= debut_valid].copy()        # sert à les tester
passe["v"] = passe["unit_sales"].clip(lower=0)            # les retours (négatifs) comptent pour 0

print(f"passé      : {len(passe):,} lignes, jusqu'au {passe['date'].max().date()}")
print(f"validation : {len(valid):,} lignes, du {debut_valid.date()} au {fin_train.date()}")

# ÉTAPE 2 : la fonction qui mesure l'erreur
resultats = []

def mesurer(nom, prediction):
    """Enregistre le RMSLE et le MAE d'une règle sur la validation."""
    vrai = valid["unit_sales"].clip(lower=0).to_numpy()
    pred = np.clip(np.asarray(prediction, dtype="float64"), 0, None)
    rmsle = np.sqrt(np.mean((np.log1p(pred) - np.log1p(vrai)) ** 2))
    mae = np.mean(np.abs(pred - vrai))
    resultats.append((nom, round(rmsle, 4), round(mae, 3)))

# ÉTAPE 3 : les règles
# Règle 0 : toujours 0
mesurer("0 partout", np.zeros(len(valid)))

# Règle 1 : la moyenne de tout le passé
moyenne_globale = passe["v"].mean()
mesurer("moyenne globale", np.full(len(valid), moyenne_globale))

# Règle 2 : la moyenne de l'ARTICLE (tous magasins confondus)
moy_article = passe.groupby("item_nbr")["v"].mean().rename("moy_article")
valid = valid.merge(moy_article, on="item_nbr", how="left")
mesurer("moyenne par article", valid["moy_article"].fillna(moyenne_globale))

# Règle 3 : la moyenne du COUPLE (magasin, article)
moy_couple = passe.groupby(["store_nbr", "item_nbr"])["v"].mean().rename("moy_couple")
valid = valid.merge(moy_couple, on=["store_nbr", "item_nbr"], how="left")
mesurer("moyenne par (magasin, article)",
        valid["moy_couple"].fillna(valid["moy_article"]).fillna(moyenne_globale))

# Règle 4 : même jour de la semaine, moyenne des 4 dernières semaines du passé
recent = passe[passe["date"] >= debut_valid - pd.Timedelta(days=28)].copy()
recent["jour"] = recent["date"].dt.dayofweek               # lundi = 0 ... dimanche = 6
moy_jour = recent.groupby(["store_nbr", "item_nbr", "jour"])["v"].mean().rename("moy_jour")

valid["jour"] = valid["date"].dt.dayofweek
valid = valid.merge(moy_jour, on=["store_nbr", "item_nbr", "jour"], how="left")
mesurer("même jour, 4 dernières semaines",
        valid["moy_jour"].fillna(valid["moy_couple"]).fillna(valid["moy_article"]).fillna(moyenne_globale))

# ÉTAPE 4 : le tableau final (plus petit = meilleur)
tableau_plancher = pd.DataFrame(resultats, columns=["règle", "RMSLE", "MAE"]).sort_values("RMSLE")
print()
print(tableau_plancher.to_string(index=False))

print("\nlignes de validation sans historique sur le couple (magasin, article) :",
      f"{valid['moy_couple'].isna().sum():,}  ({valid['moy_couple'].isna().mean():.2%})")

passé      : 12,083,439 lignes, jusqu'au 2017-07-14
validation : 164,881 lignes, du 2017-07-15 au 2017-07-30


/tmp/ipykernel_100135/3822942674.py:45: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  recent["jour"] = recent["date"].dt.dayofweek               # lundi = 0 ... dimanche = 6
/tmp/ipykernel_100135/3822942674.py:48: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  valid["jour"] = valid["date"].dt.dayofweek



                          règle  RMSLE   MAE
même jour, 4 dernières semaines 0.5899 3.824
 moyenne par (magasin, article) 0.6131 4.251
            moyenne par article 0.8028 5.819
                moyenne globale 1.0411 7.353
                      0 partout 1.8774 7.514

lignes de validation sans historique sur le couple (magasin, article) : 322  (0.20%)
